In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
with open("../data/poem.txt", encoding="utf-8") as f:
    text = f.read()

In [3]:
text = text.lower()                      
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
SEQ_LEN = 40

In [4]:
split = int(len(text) * 0.9)
train_text, val_text = text[:split], text[split:]

In [5]:
def make_windows(t):
    enc = np.array([char2idx[c] for c in t])
    X = np.stack([enc[i:i + SEQ_LEN] for i in range(len(enc) - SEQ_LEN)])
    y = enc[SEQ_LEN:]
    return X, y

In [6]:
X_train, y_train = make_windows(train_text)
X_val, y_val = make_windows(val_text)

In [7]:
model = models.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(vocab_size, 64),
    layers.SimpleRNN(256, return_sequences=False),
    layers.Dense(vocab_size, activation="softmax"),
])

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),  # clipnorm: patlayan gradyanlara karşı
    loss="sparse_categorical_crossentropy",
)

In [9]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 40, 64)              │           3,776 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ (None, 256)                 │          82,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 59)                  │          15,163 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 101,115 (394.98 KB)

 Trainable params: 101,115 (394.98 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                       restore_best_weights=True)],
)

Epoch 1/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 125s 183ms/step - loss: 2.3011 - val_loss: 2.3086
Epoch 2/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 115s 170ms/step - loss: 2.0564 - val_loss: 2.2127
Epoch 3/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 119s 175ms/step - loss: 1.9414 - val_loss: 2.1600
Epoch 4/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 113s 166ms/step - loss: 1.8545 - val_loss: 2.1068
Epoch 5/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 119s 175ms/step - loss: 1.7890 - val_loss: 2.1027
Epoch 6/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 138s 168ms/step - loss: 1.7321 - val_loss: 2.0901
Epoch 7/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 146s 174ms/step - loss: 1.6849 - val_loss: 2.0931
Epoch 8/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 143s 175ms/step - loss: 1.6430 - val_loss: 2.0697
Epoch 9/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 138s 169ms/step - loss: 1.6048 - val_loss: 2.0977
Epoch 10/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 146s 174ms/step - loss: 1.5701 - val_loss: 2.0860
Epoch 11/30
680/680 ━━━━━━━━━━━━━━━━━━━━ 138s 168ms/step - loss: 1.5389 - val_loss: 2.0905
Epoch 12

In [14]:
val_loss = min(history.history["val_loss"])
print(f"En iyi val loss: {val_loss:.3f} | perplexity: {np.exp(val_loss):.2f}")

En iyi val loss: 2.070 | perplexity: 7.92


In [15]:
def sample(probs, temperature=0.8):
    logits = np.log(np.asarray(probs, dtype="float64") + 1e-9) / temperature
    exp = np.exp(logits - logits.max())
    p = exp / exp.sum()
    return np.random.choice(len(p), p=p)

In [16]:
def generate(seed_text, n_chars=300, temperature=0.8):
    seed_text = seed_text.lower()[-SEQ_LEN:].rjust(SEQ_LEN)   # kısa seed'i boşlukla doldur
    window = [char2idx.get(c, char2idx[" "]) for c in seed_text]
    out = seed_text
    for _ in range(n_chars):
        x = np.array(window[-SEQ_LEN:])[None, :]
        probs = model.predict(x, verbose=0)[0]
        idx = sample(probs, temperature)
        out += idx2char[idx]
        window.append(idx)
    return out

In [17]:
print(generate("Stay, I said to the cut flowers.\n", temperature=0.7))

       stay, i said to the cut flowers.
that the died the hear me has lack a sabes, into ible sumber and they gain,
in the manning to the wills we man with the fields upbonts in a vaich.
i striesed in the door,
live seen, were love, the craim to know
ill to the grief.
them still leaves to forth me
and the bown
and i rever was in the land 


In [18]:
model.save("../model/poem_generate.keras")